In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
df = pd.read_csv(INPUT / 'source.csv')

In [4]:
# original Zhang / spaarw cohort; cumulative productivity includes pre-window years
# missing phd years are excluded by the phd >= 1980 rule below
frame = df.assign(dblp_id=df.dblp + df.phd_year.astype(str)).sort_values(['dblp_id', 'year'])
frame['cumpubs'] = frame.groupby('dblp_id').pubs_adj.cumsum()
frame['next_source_year'] = frame.groupby('dblp_id').year.shift(-1)
# legacy handling drops each scholar's terminal source row before windowing
eligible = frame.loc[frame.next_source_year.notna()].copy()
excluded = eligible.loc[eligible.CareerAge.eq(5) & eligible.cumpubs.lt(3), 'dblp_id']
long = eligible.loc[~eligible.dblp_id.isin(excluded) & eligible.phd_year.ge(1980)
                    & eligible.CareerAge.between(1, Y + 1)].copy()
long['age'] = long.CareerAge.astype(int) - 1
long = long.sort_values(['dblp_id', 'age'])
# missing follow-up remains NaN; observed zeros remain zero
panel = long.pivot(index='dblp_id', columns='age', values='pubs_adj').reindex(columns=range(Y + 1))
full = panel.dropna()
long['z'] = np.log(long.pubs_adj + EPS)
grouped = long.groupby('dblp_id', sort=False)
long['next_age'] = grouped.age.shift(-1)
long['q_next'] = grouped.pubs_adj.shift(-1)
long['z_next'] = grouped.z.shift(-1)
pairs = long.loc[long.next_age.eq(long.age + 1) & long.age.lt(Y),
                 ['dblp_id', 'age', 'pubs_adj', 'q_next', 'z', 'z_next']].rename(columns={'pubs_adj':'q'})
pairs['dz'] = pairs.z_next - pairs.z
pairs['state'] = pairs.q.gt(0).astype(int)
pairs['next_state'] = pairs.q_next.gt(0).astype(int)
pairs['stage'] = pd.cut(pairs.age, [-1, 0, 4, 7, Y-1], labels=['0', '1-4', '5-7', '8-20']).astype(str)

In [5]:
audit = pd.DataFrame([{'scholars':len(panel), 'rows':len(long), 'initial_values':panel[0].notna().sum(),
                       'complete_trajectories':len(full), 'transitions':len(pairs),
                       'missing_age5_scholars_retained':len(set(long.dblp_id) - set(long.loc[long.age.eq(4), 'dblp_id']))}])
long.to_csv(OUTPUT / 'long.csv', index=False)
panel.to_csv(OUTPUT / 'panel.csv')
full.to_csv(OUTPUT / 'panel_complete.csv')
pairs.to_csv(OUTPUT / 'transitions.csv', index=False)
audit.to_csv(OUTPUT / 'cohort_audit.csv', index=False)
display(audit)

,scholars,rows,initial_values,complete_trajectories,transitions,missing_age5_scholars_retained
0,2086,29119,1967,510,27033,141
